# Lecture 1 — General Linear Models & Basis Functions

**Statistisches Lernen 2** — FH Kufstein Tirol  
Professor: Johannes Schwab, PhD

---

## O que estamos aprendendo nesta aula (Resumo em 5 linhas)

Esta aula expande os horizontes da regressão linear clássica. Você aprenderá o **"truque" das Basis Functions**, que nos permite modelar curvas complexas e dados sinuosos sem perder a simplicidade matemática da regressão linear. Veremos como quebrar uma função em pedaços usando **Splines** e como usar matrizes e álgebra linear pura para encontrar os parâmetros ideais de forma analítica e exata. Ao final, você entenderá que **Linear** em General Linear Models se refere à linearidade nos *coeficientes*, não nas features — e isso abre um mundo de possibilidades.

---

## Roteiro do Notebook

1. **The Statistical Learning Problem** — O modelo $Y = f(x) + \epsilon$, Prediction vs. Inference  
2. **Linear Regression — Matrix Form** — Design Matrix e Normal Equations  
3. **Basis Functions — The Non-linear Trick** — Como modelar não-linearidade mantendo a álgebra simples  
4. **Polynomial Basis** — 1D e Multivariada  
5. **Gaussian RBF Basis** — Centers e Width  
6. **Fourier Basis** — Para dados periódicos  
7. **Splines & B-Splines** — Knots, Cox-de Boor, Compact Support, Partition of Unity  
8. **Matrix Formulation for Basis** — Resolvendo com $\hat{a} = (B^T B)^{-1} B^T y$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import BSpline

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---

## 1. The Statistical Learning Problem

### Conceito Teórico

Imagine que você quer prever o preço de uma casa a partir de sua área. Você tem dados — pares (área, preço) — mas o mundo real é bagunçado: duas casas com a mesma área podem ter preços diferentes por fatores que você não mediu (vizinhança, estado da pintura, sorte na negociação). 

O Statistical Learning Problem formaliza isso de forma elegante: existe uma relação **sistemática** entre entrada e saída, mais um **ruído irredutível**.

### Matemática

$$Y = f(\mathbf{x}) + \epsilon$$

| Símbolo | Significado |
|---------|-------------|
| $Y$ | Variável de saída (target) |
| $\mathbf{x} = (x_1, x_2, \ldots, x_p)$ | Vetor de features de entrada |
| $f$ | Função desconhecida que queremos estimar |
| $\epsilon$ | Erro aleatório com média zero, independente de $\mathbf{x}$ |

$f$ representa a **informação sistemática** que $\mathbf{x}$ carrega sobre $Y$. O nosso trabalho é **estimar $f$**.

### Why & How: Prediction vs. Inference

Há dois motivos completamente diferentes para estimar $f$:

| Objetivo | Pergunta | Exemplo |
|----------|----------|---------|
| **Prediction** | Dado um novo $\mathbf{x}$, qual será $Y$? | Prever preço de uma nova casa |
| **Inference** | Como $Y$ muda quando $x_i$ aumenta em 1 unidade? | Qual o efeito da área no preço? |

> **Insight:** Para *Prediction*, a caixa-preta pode ser opaca — só precisamos que $\hat{f}(\mathbf{x}) \approx Y$. Para *Inference*, precisamos entender a estrutura interna de $f$ — o modelo precisa ser interpretável.

In [ ]:
# Simulando o problema de aprendizado estatístico
# f verdadeira (desconhecida na prática): uma senoide
f_verdadeira = lambda x: np.sin(2 * np.pi * x)

x_all = np.linspace(0, 1, 300)

# Dados de treino com ruído
n = 20
x_treino = np.sort(np.random.rand(n))
epsilon = np.random.normal(0, 0.2, n)
y_treino = f_verdadeira(x_treino) + epsilon

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Esquerda: componentes do modelo
ax = axes[0]
ax.plot(x_all, f_verdadeira(x_all), 'k-', lw=2, label='$f(x)$ — relação sistemática')
ax.scatter(x_treino, y_treino, color='tab:blue', s=60, zorder=5, label='$Y = f(x) + \\epsilon$ (dados observados)')
for xi, yi in zip(x_treino, y_treino):
    ax.plot([xi, xi], [f_verdadeira(xi), yi], 'r-', alpha=0.4, lw=1)
ax.plot([], [], 'r-', alpha=0.6, label='$\\epsilon$ (ruído irredutível)')
ax.set_title('O modelo $Y = f(x) + \\epsilon$')
ax.set_xlabel('x')
ax.set_ylabel('Y')
ax.legend(fontsize=10)

# Direita: Prediction vs. Inference
ax = axes[1]
x_ex = np.linspace(0, 1, 200)
ax.plot(x_ex, f_verdadeira(x_ex), 'k-', lw=2, label='$f(x)$ estimada')
x_novo = 0.72
y_pred = f_verdadeira(x_novo)
ax.scatter([x_novo], [y_pred], color='red', s=150, zorder=6, label=f'Prediction: $\\hat{{Y}}({x_novo:.2f}) = {y_pred:.2f}$')
ax.annotate('Inference:\ncomo muda Y\nquando x aumenta?',
            xy=(0.3, f_verdadeira(0.3)), xytext=(0.05, -1.3),
            arrowprops=dict(arrowstyle='->', color='green'),
            color='green', fontsize=10)
ax.set_title('Prediction vs. Inference')
ax.set_xlabel('x')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---

## 2. Linear Regression — Matrix Form

### Conceito Teórico

A regressão linear diz que a relação sistemática $f$ é **linear nas features**: a saída $y$ é uma combinação ponderada das entradas. Pense nisso como uma receita: o preço de uma casa é uma certa quantia por metro quadrado mais uma quantia por quarto, mais um valor base.

### Matemática — Forma Expandida

Para cada observação $j$:

$$y_j = f(\mathbf{x}_j) + \epsilon_j = a_0 + a_1 x_j^1 + a_2 x_j^2 + \cdots + a_n x_j^n + \epsilon_j$$

Adicionando uma dimensão artificial $x_j^0 = 1$ (para absorver o bias $a_0$):

$$y_j = \mathbf{x}_j^T \mathbf{a} + \epsilon_j$$

### Matemática — Forma Compacta com a Design Matrix

Para **todos** os $K$ pontos de treino simultaneamente:

$$\mathbf{y} = X\mathbf{a} + \boldsymbol{\epsilon}$$

A **Design Matrix** $X \in \mathbb{R}^{K \times (n+1)}$ tem uma linha por observação e uma coluna por coeficiente:

$$X = \begin{pmatrix} 1 & x_1^1 & x_1^2 & \cdots & x_1^n \\ 1 & x_2^1 & x_2^2 & \cdots & x_2^n \\ \vdots & & & & \vdots \\ 1 & x_K^1 & x_K^2 & \cdots & x_K^n \end{pmatrix}$$

### Normal Equations — Solução Analítica Exata

Queremos minimizar a soma dos erros ao quadrado: $\|\mathbf{y} - X\mathbf{a}\|^2$.

Derivando em relação a $\mathbf{a}$ e igualando a zero:

$$\boxed{\hat{\mathbf{a}} = (X^T X)^{-1} X^T \mathbf{y}}$$

Esta é a **solução analítica exata** — não é uma aproximação iterativa, é álgebra linear pura!

### Why & How

A forma matricial é poderosa porque:
- Generaliza para qualquer número de features sem mudar a equação
- Permite resolver o problema de uma vez com operações de álgebra linear
- É o ponto de partida para as **Basis Functions** (Seção 3)

In [ ]:
# Dados de exemplo: relação linear com ruído
n_obs = 30
x_lin = np.sort(np.random.uniform(-2, 2, n_obs))
y_lin = 1.5 + 2.3 * x_lin + np.random.normal(0, 0.5, n_obs)

# Construindo a Design Matrix (coluna de 1s + coluna de x)
X = np.column_stack([np.ones(n_obs), x_lin])
print("Design Matrix X (primeiras 5 linhas):")
print(X[:5])
print(f"\nShape de X: {X.shape}  →  {n_obs} observações, 2 coeficientes (a0, a1)")

# Normal Equations: â = (XᵀX)⁻¹ Xᵀy
a_hat = np.linalg.solve(X.T @ X, X.T @ y_lin)
print(f"\nCoeficientes estimados:")
print(f"  a0 (intercept) = {a_hat[0]:.4f}  (verdadeiro: 1.5)")
print(f"  a1 (slope)     = {a_hat[1]:.4f}  (verdadeiro: 2.3)")

# Visualização
x_plot = np.linspace(-2.2, 2.2, 200)
y_pred_plot = a_hat[0] + a_hat[1] * x_plot

plt.scatter(x_lin, y_lin, label='Dados observados', s=50, zorder=5)
plt.plot(x_plot, y_pred_plot, 'r-', lw=2.5, label=f'$\\hat{{y}} = {a_hat[0]:.2f} + {a_hat[1]:.2f}x$ (Normal Eq.)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Regressão Linear — Normal Equations')
plt.legend()
plt.show()

In [ ]:
# Visualizando a estrutura de XᵀX e Xᵀy
print("XᵀX (Gram matrix):")
print(X.T @ X)
print("\nXᵀy:")
print(X.T @ y_lin)
print("\nSistema de equações: (XᵀX) â = Xᵀy")
print("Resolvendo com np.linalg.solve (mais estável que inverter explicitamente):")
print(f"â = {np.linalg.solve(X.T @ X, X.T @ y_lin)}")

---

## 3. Basis Functions — The Non-linear Trick

### Conceito Teórico

E se os dados não forem lineares? Imagine a trajetória de um pêndulo, ou o consumo elétrico ao longo do dia — curvas, não retas.

O **truque genial** das Basis Functions: em vez de aplicar $\mathbf{a}$ diretamente em $\mathbf{x}$, primeiro **transformamos** $\mathbf{x}$ com funções $B_m(x)$. O modelo continua **linear nos coeficientes** $a_m$ — só as features mudam!

**Analogia:** pense em um origami. Você dobra uma folha plana (transforma os dados), e depois dobras simples criam formas complexas. As dobras são as Basis Functions.

### Matemática

Para cada observação $j$, o modelo com $M$ basis functions é:

$$y_j = a_0 + \sum_{m=1}^{M} a_m B_m(x_j) + \epsilon_j$$

Onde $B_m(x)$ é a $m$-ésima basis function. Em forma compacta:

$$\mathbf{y} = B\mathbf{a} + \boldsymbol{\epsilon}$$

A matriz $B \in \mathbb{R}^{K \times M}$ substitui a Design Matrix $X$:

$$B = \begin{pmatrix} B_1(x_1) & B_2(x_1) & \cdots & B_M(x_1) \\ B_1(x_2) & B_2(x_2) & \cdots & B_M(x_2) \\ \vdots & & & \vdots \\ B_1(x_K) & B_2(x_K) & \cdots & B_M(x_K) \end{pmatrix}$$

### Why & How

**Por que funciona?** Os coeficientes $a_m$ multiplicam as funções $B_m$, e $a_m$ entram de forma linear. A não-linearidade está em $B_m(x)$, mas o sistema de equações para encontrar $\mathbf{a}$ continua **linear** — logo, a solução analítica exata via Normal Equations ainda funciona:

$$\boxed{\hat{\mathbf{a}} = (B^T B)^{-1} B^T \mathbf{y}}$$

---

## 4. Polynomial Basis

### Conceito Teórico

A Polynomial Basis é a mais intuitiva: usamos potências de $x$ como funções de base. É como dizer: "além da inclinação, também considere a curvatura, e a inflexão, e..."

### Matemática — 1D

As basis functions polinomiais de grau $M$ são:

$$B_m(x) = x^m, \quad m = 0, 1, 2, \ldots, M$$

O modelo fica:

$$y_j = a_0 + a_1 x_j + a_2 x_j^2 + \cdots + a_M x_j^M$$

A Design Matrix polinomial:

$$B_{\text{poly}} = \begin{pmatrix} 1 & x_1 & x_1^2 & \cdots & x_1^M \\ 1 & x_2 & x_2^2 & \cdots & x_2^M \\ \vdots & & & & \vdots \end{pmatrix}$$

### Why & How

**Vantagem:** simples de construir, solução exata disponível.  
**Desvantagem:** graus altos causam **overfitting** e oscilações nas bordas (*Runge's phenomenon*). Uma mudança em um ponto afeta o polinômio inteiro — sem localidade.

In [ ]:
# Dados não-lineares para demonstrar Polynomial Basis
x_nl = np.sort(np.random.uniform(-1, 1, 25))
y_nl = np.sin(3 * x_nl) + np.random.normal(0, 0.15, 25)

x_plot = np.linspace(-1, 1, 300)

def polynomial_basis_matrix(x, degree):
    """Constrói a Design Matrix polinomial: colunas são x^0, x^1, ..., x^degree."""
    return np.column_stack([x**m for m in range(degree + 1)])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
graus = [2, 5, 12]
cores = ['tab:blue', 'tab:green', 'tab:red']

for ax, grau, cor in zip(axes, graus, cores):
    # Construir B para treino
    B_treino = polynomial_basis_matrix(x_nl, grau)
    # Normal Equations
    a_hat = np.linalg.lstsq(B_treino, y_nl, rcond=None)[0]
    # Predições
    B_plot = polynomial_basis_matrix(x_plot, grau)
    y_pred = B_plot @ a_hat
    
    ax.scatter(x_nl, y_nl, s=50, color='gray', zorder=5, label='Dados')
    ax.plot(x_plot, np.sin(3 * x_plot), 'k--', alpha=0.5, label='$f$ verdadeira')
    ax.plot(x_plot, y_pred, '-', color=cor, lw=2.5, label=f'Grau {grau}')
    ax.set_title(f'Polynomial Basis — Grau {grau}')
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=9)
    ax.set_xlabel('x')

plt.suptitle('Polynomial Basis: grau baixo (underfitting) → grau alto (overfitting)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizando as próprias basis functions polinomiais
grau = 5
x_vis = np.linspace(-1, 1, 200)

plt.figure(figsize=(9, 4))
for m in range(grau + 1):
    plt.plot(x_vis, x_vis**m, label=f'$B_{m}(x) = x^{m}$')
plt.xlabel('x')
plt.ylabel('$B_m(x)$')
plt.title('Polynomial Basis Functions — cada linha é uma coluna da Design Matrix')
plt.legend(ncol=2, fontsize=9)
plt.show()

---

## 5. Gaussian RBF Basis (Radial Basis Functions)

### Conceito Teórico

Imagine colocar sinos gaussianos espalhados pelo eixo $x$, e o modelo é a soma ponderada desses sinos. Cada sino "ativa" quando $x$ está próximo de seu centro. É muito mais **local** que polinômios.

**Analogia:** pense em um time de especialistas. Cada especialista (RBF) domina uma região do espaço de entrada e contribui com sua expertise quando o ponto de consulta está na sua vizinhança.

### Matemática

A $m$-ésima Gaussian RBF é:

$$B_m(x) = \exp\left(-\frac{(x - c_m)^2}{2\sigma^2}\right)$$

| Parâmetro | Nome | Papel |
|-----------|------|-------|
| $c_m$ | **Center** | Onde o sino está centrado |
| $\sigma$ | **Width** | Quão largo é o sino (raio de influência) |

**Propriedades:**
- Cada RBF tem **suporte local** (influência decai exponencialmente)
- $\sigma$ pequeno → bases estreitas, mais flexibilidade mas mais risco de overfitting
- $\sigma$ grande → bases largas, comportamento mais suave

### Why & How

RBFs resolvem o problema de localidade do polinômio. Quando você move um parâmetro $a_m$, **apenas a vizinhança de $c_m$** é afetada. Isso é a semente do conceito de **Compact Support** que veremos em B-Splines.

In [ ]:
def rbf_basis_matrix(x, centers, sigma):
    """Constrói a Design Matrix com Gaussian RBFs."""
    return np.column_stack([
        np.exp(-(x - c)**2 / (2 * sigma**2)) for c in centers
    ])

# Centros e largura
n_centers = 8
centers = np.linspace(-1, 1, n_centers)
sigma = 0.3

# Visualizar as basis functions
x_vis = np.linspace(-1.2, 1.2, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for m, c in enumerate(centers):
    rbf = np.exp(-(x_vis - c)**2 / (2 * sigma**2))
    ax.plot(x_vis, rbf, lw=2, label=f'$c_{m}={c:.2f}$')
    ax.axvline(c, color='gray', lw=0.5, ls='--')
ax.set_title(f'Gaussian RBF Basis Functions ($\\sigma={sigma}$)')
ax.set_xlabel('x')
ax.set_ylabel('$B_m(x)$')
ax.legend(fontsize=8, ncol=2)

# Ajuste com RBF
ax = axes[1]
B_treino = rbf_basis_matrix(x_nl, centers, sigma)
a_hat = np.linalg.lstsq(B_treino, y_nl, rcond=None)[0]
B_plot = rbf_basis_matrix(x_plot, centers, sigma)
y_pred = B_plot @ a_hat

ax.scatter(x_nl, y_nl, s=50, color='gray', zorder=5, label='Dados')
ax.plot(x_plot, np.sin(3 * x_plot), 'k--', alpha=0.5, label='$f$ verdadeira')
ax.plot(x_plot, y_pred, 'tab:orange', lw=2.5, label=f'RBF fit ($M={n_centers}$, $\\sigma={sigma}$)')
for c in centers:
    ax.axvline(c, color='orange', lw=0.5, ls=':', alpha=0.5)
ax.set_title('Ajuste com Gaussian RBF Basis')
ax.set_xlabel('x')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Efeito do sigma (width) na suavidade
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sigmas = [0.1, 0.3, 1.0]

for ax, sig in zip(axes, sigmas):
    B_treino = rbf_basis_matrix(x_nl, centers, sig)
    a_hat = np.linalg.lstsq(B_treino, y_nl, rcond=None)[0]
    y_pred = rbf_basis_matrix(x_plot, centers, sig) @ a_hat
    
    ax.scatter(x_nl, y_nl, s=40, color='gray', zorder=5)
    ax.plot(x_plot, np.sin(3 * x_plot), 'k--', alpha=0.4, lw=1.5)
    ax.plot(x_plot, y_pred, 'tab:orange', lw=2.5)
    ax.set_title(f'$\\sigma = {sig}$')
    ax.set_ylim(-2, 2)
    ax.set_xlabel('x')

plt.suptitle('Efeito do Width ($\\sigma$) na Gaussian RBF: pequeno=overfit, grande=underfit', fontsize=11)
plt.tight_layout()
plt.show()

---

## 6. Fourier Basis

### Conceito Teórico

Senos e cossenos são as basis functions naturais para **dados periódicos**. O som, ondas de rádio, temperaturas sazonais — todos têm periodicidade. A Fourier Basis decompõe qualquer sinal periódico em frequências.

**Analogia:** imagine que qualquer música pode ser decomposta em notas musicais puras. Cada nota é uma frequência, e a música é a soma ponderada dessas notas. A Fourier Basis faz exatamente isso com dados.

### Matemática

Para dados com período $T$, as basis functions são:

$$B_{2k-1}(x) = \sin\left(\frac{2\pi k x}{T}\right), \quad B_{2k}(x) = \cos\left(\frac{2\pi k x}{T}\right), \quad k = 1, 2, \ldots, K$$

O modelo:

$$y_j = a_0 + \sum_{k=1}^{K} \left[ a_{2k-1} \sin\left(\frac{2\pi k x_j}{T}\right) + a_{2k} \cos\left(\frac{2\pi k x_j}{T}\right) \right] + \epsilon_j$$

### Why & How

Para dados periódicos, Fourier Basis é **ótima** em termos de representação compacta: poucas frequências capturam padrões complexos. Para dados não-periódicos, é melhor usar RBFs ou Splines.

In [ ]:
def fourier_basis_matrix(x, n_freqs, periodo=1.0):
    """Constrói a Design Matrix com Fourier Basis (senos e cossenos)."""
    cols = [np.ones_like(x)]  # termo constante a0
    for k in range(1, n_freqs + 1):
        cols.append(np.sin(2 * np.pi * k * x / periodo))
        cols.append(np.cos(2 * np.pi * k * x / periodo))
    return np.column_stack(cols)

# Dados periódicos
T = 1.0
x_per = np.sort(np.random.uniform(0, T, 40))
f_per = lambda x: 0.5 * np.sin(2 * np.pi * x) + 0.3 * np.cos(4 * np.pi * x) + 0.2 * np.sin(6 * np.pi * x)
y_per = f_per(x_per) + np.random.normal(0, 0.08, len(x_per))

x_plot_per = np.linspace(0, T, 400)

# Visualizar as basis functions
fig, axes = plt.subplots(2, 1, figsize=(11, 7))

ax = axes[0]
cores_fourier = ['tab:blue', 'tab:red', 'tab:blue', 'tab:red', 'tab:blue', 'tab:red']
estilos = ['-', '-', '--', '--', ':', ':']
labels = ['$\\sin(2\\pi x)$', '$\\cos(2\\pi x)$', '$\\sin(4\\pi x)$', '$\\cos(4\\pi x)$', '$\\sin(6\\pi x)$', '$\\cos(6\\pi x)$']
for k in range(1, 4):
    ax.plot(x_plot_per, np.sin(2*np.pi*k*x_plot_per), ls=estilos[2*k-2], color='tab:blue', lw=1.8, label=f'sin({2*k}πx)')
    ax.plot(x_plot_per, np.cos(2*np.pi*k*x_plot_per), ls=estilos[2*k-1], color='tab:red', lw=1.8, label=f'cos({2*k}πx)')
ax.set_title('Fourier Basis Functions (3 frequências)')
ax.set_xlabel('x')
ax.legend(ncol=3, fontsize=9)

ax = axes[1]
for n_freq in [1, 3, 6]:
    B_treino = fourier_basis_matrix(x_per, n_freq, T)
    a_hat = np.linalg.lstsq(B_treino, y_per, rcond=None)[0]
    y_pred = fourier_basis_matrix(x_plot_per, n_freq, T) @ a_hat
    ax.plot(x_plot_per, y_pred, lw=2, label=f'{n_freq} freq(s)')
ax.scatter(x_per, y_per, s=40, color='gray', zorder=5, label='Dados')
ax.plot(x_plot_per, f_per(x_plot_per), 'k--', alpha=0.5, lw=1.5, label='$f$ verdadeira')
ax.set_title('Ajuste com Fourier Basis — mais frequências = mais detalhes')
ax.set_xlabel('x')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---

## 7. Splines & B-Splines

### Conceito Teórico — Splines

Splines são a solução elegante para o problema de localidade dos polinômios. A ideia: **dividir o eixo $x$ em segmentos** com pontos de divisão chamados **Knots** ($\xi_1, \xi_2, \ldots$), e ajustar um polinômio separado em cada segmento, garantindo suavidade nas junções.

**Analogia perfeita:** é como montar um caminho em curva com peças de trilho de trem. Cada peça é reta (ou levemente curvada), mas as junções são suaves — você não sente o impacto ao passar de uma para outra.

### Matemática — Knots e Cubic Splines

Dados **Knots** $\xi_1 < \xi_2 < \cdots < \xi_L$ no intervalo $[a, b]$:

- Em cada segmento $[\xi_l, \xi_{l+1}]$, ajustamos um polinômio de grau $p$
- Nas junções (knots), exigimos continuidade das derivadas até ordem $p-1$

Para **Cubic Splines** ($p=3$), cada junção tem: continuidade da função, 1ª derivada e 2ª derivada — visualmente muito suave.

### Conceito Teórico — B-Splines

**B-Splines** (Basis Splines) são a representação numérica estável de Splines. Em vez de trabalhar com os polinômios por partes diretamente, usamos funções de base com **duas propriedades cruciais**:

1. **Compact Support**: cada $B_{m,p}(x)$ é exatamente zero fora de um intervalo pequeno. Mudar $a_m$ afeta **apenas a região local** — ao contrário de polinômios globais.

2. **Partition of Unity**: em qualquer ponto $x$, as B-Splines ativas somam exatamente 1:
$$\sum_{m} B_{m,p}(x) = 1$$

### Matemática — Fórmula de Cox-de Boor (Construção Recursiva)

B-Splines são definidas recursivamente. Começando com as funções constantes (grau 0):

$$B_{m,0}(x) = \begin{cases} 1 & \text{se } t_m \leq x < t_{m+1} \\ 0 & \text{caso contrário} \end{cases}$$

Para grau $p > 0$:

$$B_{m,p}(x) = \frac{x - t_m}{t_{m+p} - t_m} B_{m,p-1}(x) + \frac{t_{m+p+1} - x}{t_{m+p+1} - t_{m+1}} B_{m+1,p-1}(x)$$

Onde $t_m$ são os **knots** (sequência de nós, podendo incluir nós repetidos nas bordas).

### Why & How — Por que B-Splines são a escolha preferida?

| Propriedade | Polinômio | RBF | **B-Spline** |
|-------------|-----------|-----|-------------|
| Suporte Local (Compact Support) | Não | Aproximado | **Sim (exato)** |
| Partition of Unity | Não | Não | **Sim** |
| Suavidade controlável | Não | Indiretamente | **Sim (pelo grau)** |
| Estabilidade numérica | Baixa | Média | **Alta** |

> **Metacognição:** Ao estudar B-Splines, note como o **Compact Support** é elegante. Em vez de mudar um parâmetro e alterar o gráfico inteiro (como acontece na regressão polinomial), as Splines permitem que você ajuste uma região local dos dados sem bagunçar o resto do modelo. É o equivalente matemático a fazer uma **edição cirúrgica em um vetor**!

In [ ]:
# Construindo B-Splines do zero com Cox-de Boor
def b_spline_cox_de_boor(x, knots, m, p):
    """Avalia B_{m,p}(x) recursivamente via Cox-de Boor."""
    t = knots
    if p == 0:
        return np.where((t[m] <= x) & (x < t[m + 1]), 1.0, 0.0)
    
    left = np.zeros_like(x, dtype=float)
    right = np.zeros_like(x, dtype=float)
    
    denom_l = t[m + p] - t[m]
    if denom_l > 0:
        left = (x - t[m]) / denom_l * b_spline_cox_de_boor(x, knots, m, p - 1)
    
    denom_r = t[m + p + 1] - t[m + 1]
    if denom_r > 0:
        right = (t[m + p + 1] - x) / denom_r * b_spline_cox_de_boor(x, knots, m + 1, p - 1)
    
    return left + right


# Knots uniformes com nós repetidos nas bordas (clamped B-spline)
grau = 3  # cúbico
knots_internos = np.linspace(0, 1, 6)  # 4 nós internos
# Repetir as bordas p+1 vezes para forçar que a spline passe pelos extremos
knots = np.concatenate([[0] * grau, knots_internos, [1] * grau])
n_basis = len(knots) - grau - 1

x_vis = np.linspace(0, 1 - 1e-9, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Esquerda: as basis functions individualmente
ax = axes[0]
soma = np.zeros_like(x_vis)
for m in range(n_basis):
    bm = b_spline_cox_de_boor(x_vis, knots, m, grau)
    ax.plot(x_vis, bm, lw=2, label=f'$B_{{{m},{grau}}}$')
    soma += bm

for k in knots_internos[1:-1]:
    ax.axvline(k, color='gray', lw=0.8, ls='--', alpha=0.6)
ax.set_title(f'B-Spline Basis Functions (grau {grau}, {n_basis} bases)\nLinhas cinzas = Knots')
ax.set_xlabel('x')
ax.set_ylabel('$B_{m,3}(x)$')
ax.legend(fontsize=8, ncol=2)

# Direita: Partition of Unity
ax = axes[1]
ax.plot(x_vis, soma, 'k-', lw=3, label='$\\sum_m B_{m,3}(x)$ (Partition of Unity)')
ax.axhline(1, color='red', ls='--', lw=1.5, label='Valor esperado = 1')
ax.set_ylim(0, 1.3)
ax.set_title('Partition of Unity: $\\sum_m B_{m,p}(x) = 1$ em todo ponto')
ax.set_xlabel('x')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Demonstrando Compact Support: mudar um coeficiente afeta só a região local
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Usar scipy para B-splines (mais estável para ajuste)
from scipy.interpolate import make_lsq_spline

x_cs = np.sort(np.random.uniform(0, 1, 50))
y_cs = np.sin(2 * np.pi * x_cs) + np.random.normal(0, 0.1, 50)

knots_fit = np.linspace(0.1, 0.9, 8)
spl = make_lsq_spline(x_cs, y_cs, knots_fit, k=3)
x_spl = np.linspace(0, 1, 400)

# Ajuste original
ax = axes[0]
ax.scatter(x_cs, y_cs, s=25, color='gray', zorder=5, alpha=0.7)
ax.plot(x_spl, spl(x_spl), 'tab:blue', lw=2.5, label='B-Spline fit')
ax.plot(x_spl, np.sin(2*np.pi*x_spl), 'k--', alpha=0.4, lw=1.5, label='$f$ verdadeira')
for k in knots_fit:
    ax.axvline(k, color='orange', lw=0.8, ls=':', alpha=0.7)
ax.set_title('B-Spline fit original\n(linhas laranjas = knots internos)')
ax.legend(fontsize=9)
ax.set_xlabel('x')

# Modificando um coeficiente central — só afeta região local
ax = axes[1]
c_modificado = spl.c.copy()
idx_central = len(c_modificado) // 2
c_modificado[idx_central] += 1.0  # perturbação grande

spl_modificado = BSpline(spl.t, c_modificado, spl.k)

ax.scatter(x_cs, y_cs, s=25, color='gray', zorder=5, alpha=0.7)
ax.plot(x_spl, spl(x_spl), 'tab:blue', lw=2, ls='--', alpha=0.6, label='Original')
ax.plot(x_spl, spl_modificado(x_spl), 'tab:red', lw=2.5, label=f'Coef. {idx_central} aumentado em +1.0')
ax.axvspan(spl.t[idx_central], spl.t[idx_central + spl.k + 1], alpha=0.1, color='red', label='Região afetada')
ax.set_title('Compact Support:\nmudança local → impacto local!')
ax.legend(fontsize=9)
ax.set_xlabel('x')

plt.tight_layout()
plt.show()
print(f"Número de basis functions: {len(spl.c)}")
print(f"Coeficiente modificado: índice {idx_central} (região central)")

In [ ]:
# Comparação: graus de B-Splines (suavidade controlável)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
graus_spline = [1, 2, 3]
nomes = ['Linear (grau 1)', 'Quadrático (grau 2)', 'Cúbico (grau 3)']

for ax, k, nome in zip(axes, graus_spline, nomes):
    try:
        spl_k = make_lsq_spline(x_cs, y_cs, knots_fit, k=k)
        ax.scatter(x_cs, y_cs, s=25, color='gray', alpha=0.7, zorder=5)
        ax.plot(x_spl, spl_k(x_spl), lw=2.5, color='tab:purple')
        ax.plot(x_spl, np.sin(2*np.pi*x_spl), 'k--', alpha=0.4, lw=1.5)
        for k_pos in knots_fit:
            ax.axvline(k_pos, color='orange', lw=0.5, ls=':', alpha=0.5)
    except Exception as e:
        ax.text(0.5, 0.5, str(e), ha='center', va='center', transform=ax.transAxes)
    ax.set_title(f'B-Spline {nome}')
    ax.set_xlabel('x')
    ax.set_ylim(-1.5, 1.5)

plt.suptitle('Suavidade aumenta com o grau: grau 1 (arestas) → grau 3 (muito suave)', fontsize=11)
plt.tight_layout()
plt.show()

---

## 8. Matrix Formulation for Basis — Resolvendo Tudo com Álgebra Linear

### Conceito Teórico

Independente do tipo de Basis Function escolhida (Polinomial, RBF, Fourier, B-Spline), o processo de ajuste é sempre o mesmo: construímos a matriz $B$, e resolvemos as Normal Equations. A escolha da basis é uma decisão de design do modelo; a matemática de solução é universal.

### Matemática — O Pipeline Geral

**Passo 1:** Escolha $M$ basis functions $\{B_1, B_2, \ldots, B_M\}$

**Passo 2:** Avalie-as em cada ponto de treino para construir a matriz $B \in \mathbb{R}^{K \times M}$:

$$B_{j,m} = B_m(x_j)$$

**Passo 3:** Resolva as Normal Equations:

$$\hat{\mathbf{a}} = (B^T B)^{-1} B^T \mathbf{y}$$

**Passo 4:** Para prever um novo ponto $x^*$:

$$\hat{y}^* = \sum_{m=1}^{M} \hat{a}_m B_m(x^*) = \mathbf{b}(x^*)^T \hat{\mathbf{a}}$$

### Why & How

Este framework unificado é poderoso porque:
- Trocar o tipo de basis = trocar uma função, não o algoritmo inteiro
- Combinar bases diferentes é trivial (concatenar colunas de $B$)
- Regularização entra naturalmente: $\hat{\mathbf{a}} = (B^T B + \lambda I)^{-1} B^T \mathbf{y}$ (Ridge)

In [ ]:
# Pipeline unificado: comparar todas as basis functions no mesmo dado
x_final = np.sort(np.random.uniform(0, 2*np.pi, 40))
y_final = np.sin(x_final) + 0.5 * np.cos(2*x_final) + np.random.normal(0, 0.2, 40)

x_pred = np.linspace(0, 2*np.pi, 400)

# Todas as bases em uma função genérica
def ajustar_e_prever(B_treino, B_pred, y, lam=0.0):
    """Normal Equations (com regularização L2 opcional)."""
    BTB = B_treino.T @ B_treino
    BTy = B_treino.T @ y
    if lam > 0:
        BTB += lam * np.eye(BTB.shape[0])
    a_hat = np.linalg.solve(BTB, BTy)
    return B_pred @ a_hat

# 1. Polynomial
B_poly_tr = polynomial_basis_matrix(x_final, 6)
B_poly_pr = polynomial_basis_matrix(x_pred, 6)
y_poly = ajustar_e_prever(B_poly_tr, B_poly_pr, y_final)

# 2. RBF
centers_f = np.linspace(0, 2*np.pi, 10)
B_rbf_tr = rbf_basis_matrix(x_final, centers_f, sigma=0.6)
B_rbf_pr = rbf_basis_matrix(x_pred, centers_f, sigma=0.6)
y_rbf = ajustar_e_prever(B_rbf_tr, B_rbf_pr, y_final)

# 3. Fourier
T_f = 2 * np.pi
B_four_tr = fourier_basis_matrix(x_final, 5, T_f)
B_four_pr = fourier_basis_matrix(x_pred, 5, T_f)
y_four = ajustar_e_prever(B_four_tr, B_four_pr, y_final)

# 4. B-Spline (via scipy)
knots_f = np.linspace(0.5, 2*np.pi - 0.5, 8)
spl_final = make_lsq_spline(x_final, y_final, knots_f, k=3)

# Plotagem comparativa
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
configs = [
    (axes[0, 0], y_poly, 'Polynomial Basis (grau 6)', 'tab:blue'),
    (axes[0, 1], y_rbf, 'Gaussian RBF Basis (10 centros)', 'tab:orange'),
    (axes[1, 0], y_four, 'Fourier Basis (5 frequências)', 'tab:green'),
    (axes[1, 1], spl_final(x_pred), 'B-Spline Basis (cúbico, 8 knots)', 'tab:red'),
]

f_true = np.sin(x_pred) + 0.5 * np.cos(2 * x_pred)

for ax, y_p, titulo, cor in configs:
    ax.scatter(x_final, y_final, s=30, color='gray', zorder=5, alpha=0.7, label='Dados')
    ax.plot(x_pred, f_true, 'k--', alpha=0.4, lw=1.5, label='$f$ verdadeira')
    ax.plot(x_pred, y_p, '-', color=cor, lw=2.5, label=titulo)
    ax.set_title(titulo)
    ax.set_xlabel('x')
    ax.legend(fontsize=8)
    ax.set_ylim(-2.5, 2.5)

plt.suptitle('Pipeline Unificado: $\\hat{\\mathbf{a}} = (B^T B)^{-1} B^T \\mathbf{y}$ com diferentes Basis Functions', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Resumo das dimensões das matrizes
K = len(x_final)
print("=" * 55)
print("RESUMO DAS MATRIZES B PARA CADA BASIS")
print("=" * 55)
print(f"Dados de treino: K = {K} observações")
print()
print(f"{'Basis':<30} {'Shape de B':<20} {'# coeficientes â'}")
print("-" * 55)
print(f"{'Polynomial (grau 6)':<30} {str(B_poly_tr.shape):<20} {B_poly_tr.shape[1]}")
print(f"{'Gaussian RBF (10 centros)':<30} {str(B_rbf_tr.shape):<20} {B_rbf_tr.shape[1]}")
print(f"{'Fourier (5 freqs)':<30} {str(B_four_tr.shape):<20} {B_four_tr.shape[1]}")
print(f"{'B-Spline (cúbico, 8 knots)':<30} {str((K, len(spl_final.c))):<20} {len(spl_final.c)}")
print()
print("Em todos os casos: â = (BᵀB)⁻¹ Bᵀy")
print("A escolha da basis muda B, mas a álgebra é a mesma!")

---

## Resumo Final — O que aprendemos

| Conceito | Ideia Central | Equação Chave |
|----------|---------------|--------------|
| **Statistical Learning Problem** | Todo dado tem estrutura + ruído | $Y = f(\mathbf{x}) + \epsilon$ |
| **Linear Regression** | Modelo linear nos coeficientes | $\mathbf{y} = X\mathbf{a} + \boldsymbol{\epsilon}$ |
| **Normal Equations** | Solução exata analítica | $\hat{\mathbf{a}} = (X^T X)^{-1} X^T \mathbf{y}$ |
| **Basis Functions** | Truque: transformar $x$ antes | $y_j = \sum_m a_m B_m(x_j)$ |
| **Polynomial Basis** | Potências de $x$ | $B_m(x) = x^m$ |
| **Gaussian RBF** | Sinos locais com centro e largura | $B_m(x) = e^{-(x-c_m)^2/2\sigma^2}$ |
| **Fourier Basis** | Frequências periódicas | $\sin(2\pi k x/T), \cos(2\pi k x/T)$ |
| **B-Splines** | Polinômios locais com suporte compacto | Cox-de Boor recursivo |
| **Matrix Formulation** | Framework unificado | $\hat{\mathbf{a}} = (B^T B)^{-1} B^T \mathbf{y}$ |

---

## Exercícios para Fixar

1. **Polynomial Basis:** Adicione dados nos extremos do intervalo e observe o *Runge's phenomenon* com grau alto. Como a B-Spline se comporta no mesmo cenário?

2. **RBF Centers:** O que acontece quando os centers RBF estão muito distantes entre si? E muito próximos? Experimente com `n_centers = 3` e `n_centers = 30`.

3. **Compact Support na prática:** Na Seção 7, aumente a perturbação do coeficiente para `+5.0`. Confirme que apenas a região local do B-Spline muda. Agora faça o mesmo experimento com um modelo polinomial e observe o comportamento global.

4. **Normal Equations vs. `np.linalg.lstsq`:** Para o caso da Polynomial Basis com grau 10, compare `np.linalg.solve(B.T @ B, B.T @ y)` com `np.linalg.lstsq(B, y)`. Por que os resultados podem diferir? *(Dica: condicionamento da matriz)*

5. **Regularização:** Modifique a função `ajustar_e_prever` para aceitar `lam > 0` e observe como a curva muda para cada tipo de basis. Qual basis é mais sensível à regularização?

---

### Referências

- `visao_probabilistica.ipynb` — perspectiva probabilística (Bayes, MAP, regularização)
- `lectures/Statistisches_Lernen_2_L1.pdf` — slides originais desta aula
- Hastie, Tibshirani & Friedman: *The Elements of Statistical Learning* — Capítulos 3 e 5
- James, Witten, Hastie & Tibshirani: *An Introduction to Statistical Learning* — Capítulo 7